### Recommendation System

Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.

Evaluation:

Split the dataset into training and testing sets.
Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

### Data Preprocessing:

In [1]:
# Load dataset
import pandas as pd
df = pd.read_csv("anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [2]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [4]:
# Drop or fill missing values
anime_df = df.dropna(subset=['genre', 'rating']) 

In [5]:
# Optional: fill missing episodes with median
anime_df['episodes'] = anime_df['episodes'].replace('Unknown', None)
anime_df['episodes'] = anime_df['episodes'].astype(float)
anime_df['episodes'].fillna(anime_df['episodes'].median(), inplace=True)

C:\Users\naidu\AppData\Local\Temp\ipykernel_22360\4212198595.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anime_df['episodes'] = anime_df['episodes'].replace('Unknown', None)
C:\Users\naidu\AppData\Local\Temp\ipykernel_22360\4212198595.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anime_df['episodes'] = anime_df['episodes'].astype(float)
C:\Users\naidu\AppData\Local\Temp\ipykernel_22360\4212198595.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chaine

### Feature Extraction:

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

In [7]:
# Convert genre string to list
anime_df['genre_list'] = anime_df['genre'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])


C:\Users\naidu\AppData\Local\Temp\ipykernel_22360\3637150640.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anime_df['genre_list'] = anime_df['genre'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])


In [8]:
# One-hot encode genres
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(anime_df['genre_list'])

In [9]:
# Convert back to DataFrame
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_)

In [10]:
# Combine features (you can add rating, episodes, etc.)
features_df = pd.concat([genre_df, anime_df[['rating']].reset_index(drop=True)], axis=1)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# Compute cosine similarity between all anime
cosine_sim = cosine_similarity(features_df)

In [13]:
# Create a reverse mapping of index and anime title
indices = pd.Series(anime_df.index, index=anime_df['name']).drop_duplicates()

### Recommendation Function:

In [14]:
def recommend_anime(title, num_recommendations=5):
    if title not in indices:
        print("Anime not found in dataset.")
        return []
    
    idx = indices[title]
    
    # Get pairwise similarity scores for that anime
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first (the anime itself)
    sim_scores = sim_scores[1:num_recommendations + 1]
    
    # Get indices of similar anime
    anime_indices = [i[0] for i in sim_scores]
    
    # Return titles
    return anime_df['name'].iloc[anime_indices]


In [15]:
recommendations = recommend_anime('Naruto', 5)
print("Recommended animes similar to Naruto:\n", recommendations)

Recommended animes similar to Naruto:
 615                                    Naruto: Shippuuden
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
Name: name, dtype: object


### Evaluation:

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# For simplicity, assume you have true labels (e.g., liked/not liked) for evaluation
# Split dataset
train, test = train_test_split(anime_df, test_size=0.2, random_state=42)

# You can simulate predictions and evaluate using precision/recall if labeled data exists
# (For content-based, evaluation is often qualitative rather than numeric)

### Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

#User-based collaborative filtering:
#How it works: It identifies users with similar tastes and recommends items that these similar users liked but the target user has not yet seen. 
#Logic: "People like you also liked..."
#Example: If User A and User B both liked movies X and Y, and User A also liked movie Z, the system might recommend movie Z to User B. 
#Pros: Can be highly personalized, easy to understand. 
#Cons: Can be computationally expensive as similarity calculations are often done in real-time, and it struggles with the "cold start" problem (new users or new items). 
#Item-based collaborative filtering:
#How it works: It finds items that are similar to the items a user has already liked and recommends those similar items. 
#Logic: "Because you liked item X, you might also like item Y". 
#Example: If a user rates "The Matrix" and "Inception" highly, and the system knows these two movies are similar based on user ratings, it might recommend another similar movie, such as "Tenet". 
#Pros: More scalable and efficient because item-item similarity can be pre-calculated and reused, making it more suitable for large datasets. Less affected by a user's changed preferences over time. 
#Cons: May produce less personalized recommendations compared to user-based systems. 